In [1]:
# --- Imports ---
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [2]:
# --- Load the corrected train/test data (same files 03_baseline_model.ipynb uses) ---
train = pd.read_csv('data/processed/train_clean.csv')
test = pd.read_csv('data/processed/test_clean.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

Train shape: (58735, 68)
Test shape: (11896, 68)


In [3]:
# --- Same features/target as the Linear Regression baseline, for apples-to-apples comparison ---
baseline_features = [
    'LivingArea',
    'BedroomsTotal',
    'BathroomsTotalInteger',
    'LotSizeSquareFeet',
    'YearBuilt',
    'Latitude',
    'Longitude'
]

target = 'log_price'

X_train = train[baseline_features]
y_train = train[target]

X_test = test[baseline_features]
y_test_log = test[target]
y_test_actual = test['ClosePrice']

In [4]:
# --- Shared metric functions (same definitions as 03_baseline_model.ipynb) ---
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def mdape(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

def evaluate_model(model_name, y_test_log, y_pred_log, y_test_actual, y_pred_actual):
    r2_log = r2_score(y_test_log, y_pred_log)
    r2_dollar = r2_score(y_test_actual, y_pred_actual)
    mape_score = mape(y_test_actual, y_pred_actual)
    mdape_score = mdape(y_test_actual, y_pred_actual)

    print(f"=== {model_name} — Results ===")
    print(f"R² (log scale):     {r2_log:.4f}")
    print(f"R² (dollar scale):  {r2_dollar:.4f}")
    print(f"MAPE:               {mape_score:.2f}%")
    print(f"MdAPE:              {mdape_score:.2f}%")

    return {
        'model_name': model_name,
        'r2_log_scale': r2_log,
        'r2_dollar_scale': r2_dollar,
        'mape': mape_score,
        'mdape': mdape_score
    }

In [5]:
# --- Decision Tree Regressor (default settings first) ---
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

dt_pred_log = dt_model.predict(X_test)
dt_pred_actual = np.expm1(dt_pred_log)

dt_results = evaluate_model("Decision Tree (default)", y_test_log, dt_pred_log, y_test_actual, dt_pred_actual)

=== Decision Tree (default) — Results ===
R² (log scale):     0.8397
R² (dollar scale):  0.7426
MAPE:               18.05%
MdAPE:              11.94%


In [6]:
# --- Decision Tree Regressor with a depth cap, to check for overfitting ---
# Unconstrained trees tend to memorize train data. Comparing a capped depth
# version helps show whether default depth is overfitting.
dt_model_capped = DecisionTreeRegressor(max_depth=10, random_state=42)
dt_model_capped.fit(X_train, y_train)

dt_capped_pred_log = dt_model_capped.predict(X_test)
dt_capped_pred_actual = np.expm1(dt_capped_pred_log)

dt_capped_results = evaluate_model("Decision Tree (max_depth=10)", y_test_log, dt_capped_pred_log, y_test_actual, dt_capped_pred_actual)

=== Decision Tree (max_depth=10) — Results ===
R² (log scale):     0.8287
R² (dollar scale):  0.7497
MAPE:               19.22%
MdAPE:              13.89%


In [7]:
# --- Random Forest Regressor (the main model of interest this week) ---
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_pred_log = rf_model.predict(X_test)
rf_pred_actual = np.expm1(rf_pred_log)

rf_results = evaluate_model("Random Forest (n_estimators=100)", y_test_log, rf_pred_log, y_test_actual, rf_pred_actual)

=== Random Forest (n_estimators=100) — Results ===
R² (log scale):     0.9159
R² (dollar scale):  0.8540
MAPE:               12.92%
MdAPE:              8.77%


In [8]:
# --- Side-by-side comparison table: Linear Regression baseline vs. tree models ---
# Linear Regression numbers below are hardcoded from 03_baseline_model.ipynb's
# corrected (train-only outlier cutoff) run, for direct comparison.
linear_regression_results = {
    'model_name': 'Linear Regression A (train-only outlier cutoff)',
    'r2_log_scale': 0.4578,
    'r2_dollar_scale': -0.3780,
    'mape': 37.86,
    'mdape': 29.53
}

comparison_df = pd.DataFrame([
    linear_regression_results,
    dt_results,
    dt_capped_results,
    rf_results
])

pd.set_option('display.width', 150)
pd.set_option('display.max_colwidth', 40)

display_df = comparison_df.copy()
display_df['r2_log_scale'] = display_df['r2_log_scale'].round(4)
display_df['r2_dollar_scale'] = display_df['r2_dollar_scale'].round(2)
display_df['mape'] = display_df['mape'].round(2)
display_df['mdape'] = display_df['mdape'].round(2)

print(display_df[['model_name', 'r2_log_scale', 'r2_dollar_scale', 'mape', 'mdape']])

                                model_name  r2_log_scale  r2_dollar_scale   mape  mdape
0  Linear Regression A (train-only outl...        0.4578            -0.38  37.86  29.53
1                  Decision Tree (default)        0.8397             0.74  18.05  11.94
2             Decision Tree (max_depth=10)        0.8287             0.75  19.22  13.89
3         Random Forest (n_estimators=100)        0.9159             0.85  12.92   8.77


In [9]:
# --- Worst-prediction diagnostic for Random Forest (same approach as baseline) ---
# Checking whether Random Forest specifically fixes the overshoot pattern
# we found in Linear Regression (large errors on high-but-not-extreme homes).
pd.set_option('display.float_format', lambda x: f'{x:,.0f}')

rf_diagnostics = pd.DataFrame({
    'actual': y_test_actual.values,
    'predicted': rf_pred_actual,
})
rf_diagnostics['abs_error'] = np.abs(rf_diagnostics['actual'] - rf_diagnostics['predicted'])
rf_diagnostics['pct_error'] = rf_diagnostics['abs_error'] / rf_diagnostics['actual'] * 100

rf_worst_predictions = rf_diagnostics.sort_values('abs_error', ascending=False).head(15)
print(rf_worst_predictions)

         actual  predicted  abs_error  pct_error
2838  8,499,000  2,954,799  5,544,201         65
10540 8,500,000  3,646,743  4,853,257         57
2844  7,300,000  3,120,491  4,179,509         57
176   5,300,000  1,235,708  4,064,292         77
2970  6,700,000  2,833,039  3,866,961         58
8051  7,700,000  3,839,653  3,860,347         50
51    7,095,000  3,246,534  3,848,466         54
1737  7,550,000  3,773,489  3,776,511         50
10419 5,800,000  2,097,323  3,702,677         64
11368 6,900,000  3,241,730  3,658,270         53
2007  9,000,000  5,443,045  3,556,955         40
11800 5,750,000  2,196,823  3,553,177         62
7539  7,300,000  3,751,398  3,548,602         49
321   8,130,000  4,762,045  3,367,955         41
6627  7,595,000  4,323,910  3,271,090         43


In [12]:
# --- Save model comparison results to model_results_log.csv ---
# Rebuilt directly from the result dicts in memory (not appended from the
# existing file) so reruns of this cell always produce exactly one row per
# model, regardless of how many times it's executed.
final_results = pd.DataFrame([
    linear_regression_results,
    dt_results,
    dt_capped_results,
    rf_results
])

final_results.to_csv('model_results_log.csv', index=False)

print("Saved model_results_log.csv with 4 models")
print(final_results)

Saved model_results_log.csv with 4 models
                                model_name  r2_log_scale  r2_dollar_scale       mape      mdape
0  Linear Regression A (train-only outl...      0.457800        -0.378000  37.860000  29.530000
1                  Decision Tree (default)      0.839708         0.742602  18.053997  11.943813
2             Decision Tree (max_depth=10)      0.828688         0.749673  19.221633  13.891705
3         Random Forest (n_estimators=100)      0.915943         0.854035  12.923889   8.773831
